# 1. Импорт библиотек

In [ ]:
# Импорт основных библиотек
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Библиотеки для машинного обучения
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## Вывод (пункт 1)

Подготовлено окружение и импортированы библиотеки: сначала стандартные модули Python, затем общие библиотеки для анализа данных и визуализации, после этого — библиотеки машинного обучения из sklearn, и в конце torch для нейросетевой части. sklearn используется для классических ML-моделей (например, Logistic Regression и Random Forest), а torch — для построения и обучения нейронной сети


# 2. Загрузка и разведовательный анализ данных

**Импортируем 2 файла:**

- train — данные, на которых модель учится находить закономерности.
- test — отдельные данные, которые модель не видела во время обучения; на них проверяют качество.

**Ссылка на данные:**

https://www.kaggle.com/competitions/tech-weekend-data-science-hackathon/data

**Выбранный датасет содержит 600 000 записей в обучающем наборе и 400 000 в тестовом. Каждая запись содержит id пациента и 13 признаков:**

- Age (возраст);
- Sex (пол);
- Resting_blood_pressure (артериальное давление в состоянии покоя);
- Serum_cholestoral (уровень холестерина в крови mg/dl);
- Fasting_blood_sugar (анализ глюкозы в крови натощак (> 120 mg/dl));
- Maximum_heart_rate_achieved (максимальная частота сердечных сокращений);
- Exercise_induced_angina (стенокардия при нагрузке),
- Oldpeak (депрессия ST-сегмента);
- Slope (наклон пикового сегмента ST при физической нагрузке);
- Number_of_major_vessels (количество крупных сосудов (0-3), окрашенных с помощью флюороскопии);
- Resting_electrocardiographic_results (результаты электрокардиографических исследований в покое 0, 1, 2);
- Thal (3 = норма; 6 = фиксированный дефект; 7 = обратимый дефект);
- Chest_bin (боль в груди от 1 до 4).

In [ ]:
# Загрузка тренировочных данных
train = pd.read_csv('train.csv')

# Первые 5 строк обучающего набора данных
print("Обучающий набор данных:")
display(train.head())

In [ ]:
# Загрузка тествоых данных
test = pd.read_csv('test.csv')

# Первые 5 строк тестового набора данных
print("Набор тестовых данных:")
display(test.head())

In [ ]:
print("Статистическое описание тестового набора данных:")
test.describe()

In [ ]:
# Общая информация о тестовом датасете
print("Информация о тестовомнаборе данных:")
test.info()

In [ ]:
# Проверим пропущенные значения в тестовом наборе данных
print("Пропущенные значения в тестовом наборе данных:")
print(test.isnull().sum())

In [ ]:
print("Статистическое описание обучающего набора данных:")
train.describe()

In [ ]:
# Общая информация об обучающем датасете
print("Информация об обучающем наборе данных:")
train.info()

In [ ]:
# Проверка пропущенных значений в обучающем наборе данных
print("Пропущенные значения в обучающем наборе данных:")
print(train.isnull().sum())

## Оценка текущей выполненной части

Сейчас выполнено:
- Загрузка `train.csv` и `test.csv`, просмотр первых строк.
- Базовый EDA: `info()`, `describe()`, проверка пропусков `isnull().sum()`.

Пока **не хватает** (по критериям и для дальнейшего моделирования):
- Корреляционный анализ (heatmap/диаграммы рассеяния).
- Оценка выбросов (boxplot/"ящик с усами", хотя бы для ключевых числовых признаков).
- Проверка/обработка дублей (и базовая проверка явных невалидных значений/диапазонов).
- Явная проверка баланса целевого класса (`class`).
- Раздел “Предобработка данных” пока пустой (кодирование категориальных, масштабирование, разбиение на train/valid и т.п.).

## Мини-выводы по данным (на основе текущего EDA)

- Данные крупные: **600 000** строк в train и **400 000** в test, что хорошо для обучения моделей и стабильной оценки качества.
- Пропусков **нет**: во всех столбцах `train` и `test` показано **0** пропущенных значений.
- Типы признаков уже числовые (`int64`/`float64`). При этом часть признаков по смыслу **категориальные**, хоть и закодированы числами (например: `sex`, `fasting_blood_sugar`, `resting_electrocardiographic_results`, `exercise_induced_angina`, `slope`, `number_of_major_vessels`, `thal`). Их стоит обрабатывать как категориальные/порядковые, чтобы модель не “придумывала” ложную метрику расстояния.
- По описательной статистике train/test похожи (средние/разбросы близки) — явного сдвига распределений между наборами по базовым метрикам не видно.
- Целевая переменная `class` бинарная (0/1). По `mean(class) ≈ 0.444` доля положительного класса около **44%**, то есть дисбаланс **умеренный** (не экстремальный), но лучше всё равно проверить `value_counts()`.
- Есть признаки с потенциально “широкими хвостами”/выбросами (например, `serum_cholestoral`, `resting_blood_pressure`, `oldpeak`) — это нужно подтвердить boxplot’ами и при необходимости использовать robust-скейлинг/клиппинг.
- В `oldpeak` встречаются **отрицательные** значения (примерно до -0.8 по `describe()`) — это не обязательно ошибка (зависит от определения признака/генерации датасета), но стоит проверить распределение и крайние значения.


## Вывод (пункт 2)

- Данные успешно загружены и проверены базовые свойства: типы, размеры, описательная статистика.
- Определены целевой столбец и список признаков для дальнейшего анализа и построения моделей.


# 3. Анализ данных

In [ ]:
# 3.1 Базовые проверки: дубликаты, баланс классов, базовая валидация диапазонов

# 1) Дубликаты
print("Дубликаты строк в train:", train.duplicated().sum())
print("Дубликаты ID в train:", train["ID"].duplicated().sum())

# 2) Баланс целевого класса
class_counts = train["class"].value_counts().sort_index()
class_share = train["class"].value_counts(normalize=True).sort_index()
print("\nРаспределение целевого класса (кол-во):\n", class_counts)
print("\nРаспределение целевого класса (доля):\n", class_share)

plt.figure(figsize=(4,3))
ax = sns.countplot(x="class", data=train)
ax.set_title("Баланс классов (train)")
ax.set_xlabel("class")
ax.set_ylabel("count")
plt.show()

# 3) Базовая проверка диапазонов категориальных признаков
range_checks = {
    "sex": (0, 1),
    "fasting_blood_sugar": (0, 1),
    "resting_electrocardiographic_results": (0, 2),
    "exercise_induced_angina": (0, 1),
    "slope": (1, 3),
    "number_of_major_vessels": (0, 3),
    "thal": (3, 7),
    "chest": (1, 4),  # по описанию проекта: 1..4
}

bad_values = {}
for col, (lo, hi) in range_checks.items():
    mask = (train[col] < lo) | (train[col] > hi)
    bad_values[col] = int(mask.sum())

print("\nКоличество значений вне ожидаемых диапазонов (train):")
for col, n_bad in bad_values.items():
    print(f"- {col}: {n_bad}")


## 3.2 Корреляционный анализ (числовые признаки)

Ниже считаем корреляции **только по числовым признакам** на исходных данных. Для категориальных признаков в виде “кодированных чисел” корреляция Пирсона может вводить в заблуждение, поэтому для них лучше смотреть распределения по классам.


In [ ]:
# Корреляционная матрица (по числовым признакам)
num_features = [
    "age",
    "resting_blood_pressure",
    "serum_cholestoral",
    "maximum_heart_rate_achieved",
    "oldpeak",
]

corr_df = train[num_features + ["class"]].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Корреляции (числовые признаки + class)")
plt.show()

# Топ признаков по абсолютной корреляции с class
corr_with_target = corr_df["class"].drop("class").abs().sort_values(ascending=False)
print("Топ числовых признаков по |corr| с class:")
print(corr_with_target)

# Более корректная визуализация для бинарной цели: boxplot признака по классам
top_k = corr_with_target.index[:3]
plt.figure(figsize=(12, 3))
for i, col in enumerate(top_k, start=1):
    plt.subplot(1, 3, i)
    sns.boxplot(x="class", y=col, data=train, showfliers=False)
    plt.title(f"{col} vs class")
plt.tight_layout()
plt.show()


## 3.3 Оценка выбросов (boxplot на исходных шкалах)

Boxplot’ы ниже строятся **до масштабирования**: так “выбросы” видны в исходных единицах и не путаются с отрицательными значениями, которые могли бы появиться просто из-за `StandardScaler`.


In [ ]:
# Boxplot для числовых признаков (сырые значения)

plt.figure(figsize=(14, 8))
for i, col in enumerate(num_features, start=1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=train[col], color="#8ecae6")
    plt.title(col)
plt.tight_layout()
plt.show()

# Дополнительно: распределения oldpeak (есть отрицательные значения)
plt.figure(figsize=(8, 3))
sns.histplot(train["oldpeak"], bins=50, kde=True)
plt.title("Распределение oldpeak (train)")
plt.show()


## Что улучшено по сравнению с `Heart_Disease_example.ipynb`

- Анализ данных вынесен **до** любых преобразований (one-hot/масштабирование), поэтому выводы про распределения и выбросы делаются **в исходной шкале** и не путаются с отрицательными значениями после `StandardScaler`.
- Для корреляций используется аккуратный вариант: считаем корреляции **по числовым** признакам + `class` (а для бинарной цели вместо scatter `feature` vs `class` добавлены **boxplot по классам**, что обычно информативнее).
- Добавлены проверки качества данных, которых не хватало в примере: **дубликаты**, **баланс классов**, **проверка значений категориальных признаков на допустимые диапазоны**.


In [ ]:
# 4.0 Фильтрация/исправление невалидных значений (train/test)
# Критерий: "фильтрация невалидных значений".
# Стратегия: значения вне ожидаемых диапазонов переводим в NaN, дальше они будут обработаны импутером в пайплайне.

train_clean = train.copy()
test_clean = test.copy()

# Ожидаемые диапазоны
valid_ranges = {
    "sex": (0, 1),
    "fasting_blood_sugar": (0, 1),
    "resting_electrocardiographic_results": (0, 2),
    "exercise_induced_angina": (0, 1),
    "slope": (1, 3),
    "number_of_major_vessels": (0, 3),
    "thal": (3, 7),
    # chest по заданию 1..4, но в данных он float; оставляем как числовой и тоже ограничим диапазоном
    "chest": (1, 4),
}

# Какие признаки ожидаем как целочисленные категории
int_like = {
    "sex",
    "fasting_blood_sugar",
    "resting_electrocardiographic_results",
    "exercise_induced_angina",
    "slope",
    "number_of_major_vessels",
    "thal",
}


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col, (lo, hi) in valid_ranges.items():
        if col not in df.columns:
            continue

        # для категориальных int-like аккуратно округляем
        if col in int_like:
            df[col] = np.rint(df[col]).astype("float64")

        # значения вне диапазона -> NaN
        mask_bad = (df[col] < lo) | (df[col] > hi)
        df.loc[mask_bad, col] = np.nan

    return df


train_clean = clean_df(train_clean)
test_clean = clean_df(test_clean)

print("NAs after cleaning (train):")
print(train_clean.isna().sum().sort_values(ascending=False).head(10))


## Вывод (пункт 3)

- Проведён анализ распределений и связей: проверены дубликаты/баланс классов, построены корреляции (heatmap) и оценены выбросы (boxplot).
- Выделены признаки, потенциально связанные с целевой переменной, и сформировано понимание, какие преобразования понадобятся на этапе предобработки.


# 4. Предобработка данных

Дальше — подготовка признаков для моделей (кодирование категориальных, масштабирование числовых, разбиение на train/valid) через пайплайны, чтобы не было рассинхронизации train/test и утечки данных.


In [ ]:
# 4.1 Подготовка признаков и разбиение на train/val

TARGET_COL = "class"
DROP_COLS = ["ID"]

# Категориальные признаки
# Важно: `chest` в данных выглядит как float с большим числом уникальных значений,
# поэтому One-Hot для него раздувает матрицу признаков. Оставляем `chest` как числовой.
CAT_FEATURES = [
    "sex",
    "fasting_blood_sugar",
    "resting_electrocardiographic_results",
    "exercise_induced_angina",
    "slope",
    "number_of_major_vessels",
    "thal",
]

# Числовые признаки
NUM_FEATURES = [
    "age",
    "resting_blood_pressure",
    "serum_cholestoral",
    "maximum_heart_rate_achieved",
    "oldpeak",
    "chest",
]

# Используем очищенные данные
X = train_clean.drop(columns=[TARGET_COL] + DROP_COLS)
y = train_clean[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)

# 4.2 Трансформеры
cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # Делаем OHE разреженным, чтобы не раздувать память.
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]
)

num_pipe_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

num_pipe_noscale = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Для Logistic Regression и Neural Network (со scaler)
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", num_pipe_scaled, NUM_FEATURES),
        ("cat", cat_pipe, CAT_FEATURES),
    ],
    remainder="drop",
)

# Для RandomForest (без scaler)
preprocess_noscale = ColumnTransformer(
    transformers=[
        ("num", num_pipe_noscale, NUM_FEATURES),
        ("cat", cat_pipe, CAT_FEATURES),
    ],
    remainder="drop",
)

# 4.3 Подготовленные матрицы
# Для sklearn-моделей удобно оставлять разреженную матрицу (экономит память).
X_train_prepared = preprocess_scaled.fit_transform(X_train)
X_val_prepared = preprocess_scaled.transform(X_val)

print("\nAfter preprocess_scaled:")
print("X_train_prepared shape:", X_train_prepared.shape)
print("X_val_prepared shape:", X_val_prepared.shape)
print("Matrix type:", type(X_train_prepared))

# Список итоговых фич (удобно для интерпретации и отладки)
feature_names = preprocess_scaled.get_feature_names_out()
print("\nTotal features after OHE:", len(feature_names))

# Для нейросети часто нужен dense-массив.
# Важно: переводить 600k строк в dense может быть тяжело по памяти.
# Поэтому ниже — безопасный вариант: готовим dense только для val и небольшого сэмпла train.
X_val_dense = X_val_prepared.toarray() if hasattr(X_val_prepared, "toarray") else X_val_prepared

sample_n = 50000  # можно увеличить/уменьшить при необходимости
X_train_dense = (
    X_train_prepared[:sample_n].toarray()
    if hasattr(X_train_prepared, "toarray")
    else X_train_prepared[:sample_n]
)
y_train_dense = y_train.iloc[:sample_n]

print("\nDense matrices for NN:")
print("X_train_dense:", X_train_dense.shape, "y_train_dense:", y_train_dense.shape)
print("X_val_dense:", X_val_dense.shape, "y_val:", y_val.shape)


## Готовые пайплайны для моделей

Ниже — заготовки пайплайнов (предобработка + модель) для Logistic Regression и RandomForest. Для нейросети можно использовать `X_train_prepared` / `X_val_prepared` из предыдущей ячейки.


In [ ]:
pipe_lr = Pipeline(
    steps=[
        ("prep", preprocess_scaled),
        ("model", LogisticRegression(max_iter=2000, n_jobs=-1)),
    ]
)

pipe_rf = Pipeline(
    steps=[
        ("prep", preprocess_noscale),
        ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ]
)

print("Pipelines created:", pipe_lr is not None, pipe_rf is not None)


## Вывод (пункт 4)

- Проведена очистка/фильтрация невалидных значений и сформированы `train_clean`/`test_clean` для дальнейшей работы.
- Собрана воспроизводимая схема предобработки через `ColumnTransformer`/`Pipeline` (кодирование категорий + масштабирование числовых для моделей, которым это нужно).


# 5. Обучение моделей

Ниже обучаем и сравниваем 3 модели: **Logistic Regression**, **Random Forest**, **Neural Network (PyTorch)**. Оценка проводится на валидационной выборке `X_val / y_val`.


In [ ]:
# Общие функции для оценки моделей

def evaluate_sklearn_model(model, X_tr, y_tr, X_te, y_te, title: str):
    model.fit(X_tr, y_tr)

    # Вероятности нужны для ROC-AUC
    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_te, y_proba)
    print(f"{title} | ROC-AUC: {auc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_te, y_pred, digits=4))

    ConfusionMatrixDisplay.from_predictions(y_te, y_pred, values_format="d")
    plt.title(f"Confusion matrix: {title}")
    plt.show()

    RocCurveDisplay.from_predictions(y_te, y_proba)
    plt.title(f"ROC curve: {title}")
    plt.show()

    return auc


## 5.1 Logistic Regression

Логистическая регрессия чувствительна к масштабу признаков, поэтому используем `pipe_lr` (в нём `StandardScaler` для числовых и One-Hot для категориальных).


In [ ]:
auc_lr = evaluate_sklearn_model(
    pipe_lr,
    X_train,
    y_train,
    X_val,
    y_val,
    title="Logistic Regression",
)


## 5.2 Random Forest

Случайный лес не требует масштабирования, поэтому используем `pipe_rf` (OHE для категориальных + без `StandardScaler`).


In [ ]:
auc_rf = evaluate_sklearn_model(
    pipe_rf,
    X_train,
    y_train,
    X_val,
    y_val,
    title="Random Forest",
)


## 5.3 Neural Network (PyTorch)

Ниже — простая полносвязная сеть (MLP) на PyTorch. Она обучается на подготовленных матрицах `X_train_dense / y_train_dense` и проверяется на `X_val_dense / y_val`.

Если PyTorch не установлен, установи его (CPU-версию) в активном окружении `.venv`.


In [ ]:
# Импорт PyTorch
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
set_seed(42)


In [ ]:
# Даталоадеры

Xtr = torch.tensor(X_train_dense, dtype=torch.float32)
ytr = torch.tensor(y_train_dense.values, dtype=torch.float32).view(-1, 1)

Xva = torch.tensor(X_val_dense, dtype=torch.float32)
yva = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=1024, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=4096, shuffle=False)

input_dim = Xtr.shape[1]
print("Input dim:", input_dim)


In [ ]:
# Модель (MLP)

class MLP(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


model = MLP(input_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)


In [ ]:
# Обучение + метрики

def predict_proba_torch(model, loader):
    model.eval()
    probs = []
    ys = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().ravel()
            probs.append(p)
            ys.append(yb.numpy().ravel())
    return np.concatenate(probs), np.concatenate(ys)


epochs = 10
history = {"train_loss": [], "val_auc": []}

for epoch in range(1, epochs + 1):
    model.train()
    running = 0.0
    n = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running += loss.item() * xb.size(0)
        n += xb.size(0)

    train_loss = running / n

    val_proba, val_y = predict_proba_torch(model, val_loader)
    val_auc = roc_auc_score(val_y, val_proba)

    history["train_loss"].append(train_loss)
    history["val_auc"].append(val_auc)

    print(f"Epoch {epoch:02d}/{epochs} | train_loss={train_loss:.4f} | val_auc={val_auc:.4f}")

# Графики обучения
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], marker="o")
plt.title("Train loss")
plt.xlabel("epoch")

plt.subplot(1, 2, 2)
plt.plot(history["val_auc"], marker="o")
plt.title("Val ROC-AUC")
plt.xlabel("epoch")
plt.ylim(0.0, 1.0)
plt.tight_layout()
plt.show()

# Итоговая оценка
val_pred = (val_proba >= 0.5).astype(int)
auc_nn = roc_auc_score(val_y, val_proba)

print("\nNeural Network (PyTorch) | ROC-AUC:", auc_nn)
print("\nClassification report:")
print(classification_report(val_y.astype(int), val_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(val_y.astype(int), val_pred, values_format="d")
plt.title("Confusion matrix: Neural Network (PyTorch)")
plt.show()

RocCurveDisplay.from_predictions(val_y, val_proba)
plt.title("ROC curve: Neural Network (PyTorch)")
plt.show()


## Вывод (пункт 5)

- Обучены 3 модели (LogReg, RandomForest, MLP на PyTorch) и получены метрики качества на валидации (в т.ч. `ROC-AUC`).
- Построены графики/кривые и матрицы ошибок, что позволяет сравнить модели не только по числу, но и по типам ошибок.


# 6. Сравнение моделей

Сравним модели по метрике **ROC-AUC** на валидационной выборке.


In [ ]:
results = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Random Forest",
            "Neural Network (PyTorch)",
        ],
        "roc_auc": [auc_lr, auc_rf, auc_nn],
    }
).sort_values("roc_auc", ascending=False)

display(results)

plt.figure(figsize=(6, 3))
sns.barplot(data=results, x="roc_auc", y="model", orient="h")
plt.xlim(0.0, 1.0)
plt.title("ROC-AUC на validation")
plt.xlabel("ROC-AUC")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Вывод по сравнению моделей

- Сравнение на валидационной выборке показало, что **наибольшее значение ROC-AUC** имеет модель, которая находится **на 1‑й строке таблицы `results`** (после сортировки по `roc_auc`).
- Разница в ROC-AUC между моделями показывает, насколько лучше/хуже модель **разделяет классы** (чем выше ROC-AUC, тем лучше качество ранжирования).
- Для дальнейшего этапа (**инференс на `test.csv`**) логично выбрать **лучшую по ROC-AUC модель** из таблицы `results` и дообучить её на всём `train`.

Дополнительно:
- Если ROC-AUC у моделей близкий, можно выбирать модель, которая проще в эксплуатации (скорость/интерпретируемость): например, Logistic Regression часто проще и быстрее, Random Forest может быть тяжелее, нейросеть требует больше настроек.


## Вывод (пункт 6)

- Сведены результаты моделей в единую таблицу и визуально сравнен `ROC-AUC`.
- По валидации выбрана модель‑лидер; именно её логика далее используется для финального обучения и инференса.


# 7. Инференс на `test.csv` и подготовка `submission.csv`

Ниже выбираем лучшую модель по `roc_auc` (из таблицы `results`), дообучаем её на всём `train`, делаем предсказания на `test.csv` и сохраняем файл `submission.csv` в формате, как в `sample_submission.csv`.


In [ ]:
# Выбор лучшей sklearn-модели по таблице results
# (PyTorch модель здесь не используем для submission, т.к. нужен отдельный препроцессинг/инференс.)

best_row = results.iloc[0]
best_model_name = best_row["model"]
print("Best model by ROC-AUC:", best_model_name)

if best_model_name == "Logistic Regression":
    best_pipe = pipe_lr
elif best_model_name == "Random Forest":
    best_pipe = pipe_rf
else:
    # если вдруг наверху окажется нейросеть, берём лучшую sklearn-модель
    print("Top model is PyTorch NN. For submission we use best sklearn model instead.")
    best_pipe = pipe_lr if auc_lr >= auc_rf else pipe_rf
    print("Chosen sklearn model:", "Logistic Regression" if best_pipe is pipe_lr else "Random Forest")

# Обучаем на всём train (очищенные данные)
X_full = train_clean.drop(columns=[TARGET_COL] + DROP_COLS)
y_full = train_clean[TARGET_COL]

best_pipe.fit(X_full, y_full)
print("Trained best_pipe on full train.")


In [ ]:
# Предсказание на test и сохранение submission.csv

X_test = test_clean.drop(columns=DROP_COLS)

# Вероятность класса 1
test_proba = best_pipe.predict_proba(X_test)[:, 1]

# Вариант как в sample_submission.csv: 0/1 по порогу 0.5
test_pred = (test_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": test_clean["ID"].values,
    "class": test_pred,
})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())


## Вывод (пункт 7)

- Выполнен инференс на `test.csv`: выбрана лучшая модель по `roc_auc`, дообучена на всём `train` и сформирован `submission.csv` в формате `ID,class`.
- Дополнительно сохранены артефакты модели (пайплайн/веса), чтобы инференс можно было повторить без переобучения.


# 8. Краткое содержание научной статьи (мини)

Статья рассматривает применение моделей машинного обучения для прогнозирования риска сердечно‑сосудистых заболеваний по клиническим признакам.

**Ключевые идеи:**
- **Цель**: построить модель, которая по признакам пациента оценивает вероятность наличия/риска ССЗ.
- **Подход**: подготовка данных, анализ признаков, обучение нескольких моделей и сравнение по метрикам качества.
- **Практический вывод**: качество зависит от корректной предобработки, работы с дисбалансом/выбросами и подбора модели; интерпретируемые модели (например, логистическая регрессия) могут быть полезны для объяснимости, а более сложные модели иногда дают прирост качества.

(Этот блок можно расширить: добавить 3–5 абзацев про датасет статьи, используемые алгоритмы и итоговые метрики/выводы авторов.)


In [ ]:
# 7.1 Сохранение лучшей модели (sklearn) и весов PyTorch

# сохраняем выбранную sklearn-модель (вместе с препроцессингом внутри pipeline)
joblib.dump(best_pipe, "best_model.joblib")
print("Saved sklearn model to best_model.joblib")

# сохраняем веса PyTorch-модели (если она обучалась)
try:
    torch.save(model.state_dict(), "pytorch_model.pt")
    print("Saved PyTorch weights to pytorch_model.pt")
except Exception as e:
    print("PyTorch weights were not saved:", e)


## Вывод (пункт 8)

- Кратко зафиксированы цель и общий подход статьи: подготовка данных → обучение нескольких моделей → сравнение по метрикам.
- Для отчёта этого достаточно как мини‑конспекта; при желании можно расширить блок конкретными результатами/метриками из статьи.
